<a href="https://colab.research.google.com/github/DhimanTarafdar/AAA/blob/main/Cat_vs_Dog_Classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Import Libraries**

In [1]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import transforms
from torch.utils.data import Dataset, DataLoader, Subset
from PIL import Image
import kagglehub
import matplotlib.pyplot as plt

# **Download Dataset**

In [2]:
path = kagglehub.dataset_download("salader/dogsvscats")
print("path", path)

Using Colab cache for faster access to the 'dogsvscats' dataset.
path /kaggle/input/dogsvscats


# **Setup**

In [3]:
torch.manual_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using Device: {device}")

Using Device: cuda


# **Path Setup**

In [4]:
TRAIN_PATH = os.path.join(path, "train")
VAL_PATH   = os.path.join(path, "test")

print("Train Path:", TRAIN_PATH)
print("Val Path  :", VAL_PATH)

Train Path: /kaggle/input/dogsvscats/train
Val Path  : /kaggle/input/dogsvscats/test


# **Image Transformation**

In [5]:
transform = transforms.Compose(
    [
        transforms.Resize((128, 128)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.5]*3, std=[0.5]*3)
    ]
)

# Image Transformation Pipeline

`transforms.Compose()` একটা pipeline বানায় — image একটার পর একটা step এর মধ্য দিয়ে যায়।

## ৩টা step:

- **Resize((128,128))** → যেকোনো size এর image কে 128×128 pixel এ convert করে, কারণ CNN এ সব input এর size same হতে হয়

- **ToTensor()** → PIL image (0-255) কে PyTorch tensor এ convert করে এবং pixel value কে 0.0-1.0 range এ নিয়ে আসে

- **Normalize(mean=[0.5]*3, std=[0.5]*3)** → pixel value কে 0.0-1.0 থেকে -1.0 থেকে +1.0 range এ নিয়ে আসে
  - formula: `(pixel - 0.5) / 0.5`
  - `*3` মানে RGB তিনটা channel এর জন্যই একই value

**কেন Normalize করি?** → model training এ gradient stable থাকে, faster converge করে

# **Custom Dataset**

In [9]:
class BinaryClassification(Dataset):
    def __init__(self, root_dir, transform=None):
        super().__init__()

        self.samples = []
        self.transform = transform

        self.classes = sorted([d for d in os.listdir(root_dir)
                               if os.path.isdir(os.path.join(root_dir, d))])

        self.class_to_idx = {cls_name: idx for idx, cls_name in enumerate(self.classes)}

        for class_name in self.classes:
            class_path = os.path.join(root_dir, class_name)

            for img_name in os.listdir(class_path):
                img_path = os.path.join(class_path, img_name)

                if os.path.isfile(img_path):
                    label = self.class_to_idx[class_name]
                    self.samples.append((img_path, label))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        image = Image.open(img_path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        return image, label

# Custom Dataset Class — BinaryClassification

PyTorch এ নিজের data load করতে হলে `Dataset` class কে inherit করে নিজের class বানাতে হয়।

> Plant disease এ class এর নাম ছিল `MultiClassClassfication` কারণ সেখানে 38টা class ছিল।
> Cat vs Dog এ মাত্র 2টা class (cat, dog) তাই নাম দিলাম `BinaryClassification`।

---

## `__init__()` — class তৈরির সময় যা যা হয়

**Step 1 — খালি list এবং transform রেখে দাও**
- `self.samples = []` → এখানে (image_path, label) pair জমা হবে
- `self.transform = transform` → image এ কী কী করতে হবে সেটা রেখে দিলাম

**Step 2 — সব class এর নাম বের করো**
- `root_dir` এর ভেতরে যত folder আছে সেগুলোই class — এখানে `cat` এবং `dog`
- `sorted()` দিয়ে alphabetically সাজানো হলো — তাই `cat → 0`, `dog → 1`

**Step 3 — প্রতিটা class কে একটা number দাও**
- CNN সরাসরি string বোঝে না, তাই class এর নামকে unique number এ convert করা হলো
- result: `{'cat': 0, 'dog': 1}`

**Step 4 — সব image এর path আর label একসাথে জমা করো**
- প্রতিটা class folder এ যাও → সেই folder এর প্রতিটা file দেখো → নিশ্চিত হও এটা file না folder → class নাম কে number এ convert করো → list এ জমা করো
- শেষে `self.samples` এ থাকে: `[('root/cat/img1.jpg', 0), ('root/dog/img4.jpg', 1), ...]`

---

## `__len__()` — dataset এ মোট কতটা image আছে
- DataLoader এই method কে call করে জানে যে loop কতবার চালাতে হবে

---

## `__getitem__()` — index দিলে একটা image আর তার label দাও
- `self.samples[idx]` থেকে path আর label নাও
- disk থেকে image পড়ো, `.convert("RGB")` দিয়ে নিশ্চিত করো সব image 3 channel এ আছে
- transform apply করো (resize, tensor, normalize)
- DataLoader যখন batch বানায়, সে বারবার এই method কে আলাদা আলাদা index দিয়ে call করে

---

## সহজ ভাষায় পুরো flow

```
root_dir/
├── cat/        ← class 0
│   ├── img1.jpg
│   └── img2.jpg
└── dog/        ← class 1
    └── img3.jpg

__init__    → সব image এর path + label একটা list এ জমা করো
__len__     → বলো list এ কতটা আছে
__getitem__ → index দিলে সেই image টা পড়ো, transform করো, return করো
```